# Ch. 5 · Atelier facultatif — Un LLM qui cite ses sources, et sa vérification

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/benlahmar/IASciences/blob/main/notebooks/ch05-atelier-llm.ipynb)

**IA pour les sciences — M1.** Brancher un LLM sur la recherche TF-IDF, puis contrôler automatiquement citations et nombres.

**Durée indicative :** 1 h à 1 h 30 en autonomie  
**Statut :** atelier facultatif hors volume horaire. Fonctionne **sans clé API** grâce à des réponses simulées.

**Objectifs**

- Construire un prompt RAG avec contexte, contraintes et format.
- Appeler un LLM (facultatif : clé API) ou utiliser des réponses simulées.
- Détecter automatiquement les citations inventées et les nombres absents des sources.
- Mesurer la variabilité de réponses répétées.

**Prérequis :** TP du chapitre 5 (`ch05-mini-rag.ipynb`).

> **Mode d’emploi.** Exécutez les cellules dans l’ordre. Les cellules marquées **✏️ Exercice** contiennent des `...` à compléter : une cellule non complétée lève une erreur, c’est normal. Les corrigés sont repliés sous **« Voir le corrigé »** : cherchez d’abord ! Les cellules suivantes ne dépendent jamais du résultat d’un exercice.

## 0. Règles d’usage

Ne transmettez **jamais** de données personnelles, confidentielles ou non publiées à un service externe sans autorisation.
Ici, le corpus est fictif. Un appel à une API payante est **facultatif** : sans clé, le notebook utilise des réponses enregistrées
qui contiennent volontairement des erreurs typiques.

In [ ]:
MOTS_VIDES = ('a au aux avec ce ces cette comment d de des du dans elle en est et il ils l la le les leur lors lorsque '
              'ne ni on ou par pas pour qu que quel quelle quels quelles qui sa se ses son sont sous sur un une '
              'faut doit peut quoi').split()
documents = [
 {"id": "BIO-01", "source": "Note de laboratoire biologie",
  "texte": "Une classification d'espèces doit être évaluée sur des échantillons indépendants, prélevés sur des individus différents de ceux de l'entraînement."},
 {"id": "BIO-02", "source": "Protocole de culture cellulaire",
  "texte": "Les cultures sont incubées à 37 °C sous 5 % de CO2 ; un effet de lot apparaît lorsque les plaques proviennent de séries différentes."},
 {"id": "CHIM-01", "source": "Note de laboratoire chimie",
  "texte": "La solubilité dépend notamment de la température et du solvant ; elle est mesurée en mg/L à 25 °C."},
 {"id": "CHIM-02", "source": "Fiche de spectroscopie",
  "texte": "Un spectre infrarouge présente des bandes d'absorption caractéristiques ; la ligne de base doit être corrigée avant comparaison."},
 {"id": "PHY-01", "source": "Note de laboratoire physique",
  "texte": "Les mesures répétées permettent d'estimer la dispersion expérimentale et l'incertitude de type A."},
 {"id": "PHY-02", "source": "Cahier d'expérience thermique",
  "texte": "La température décroît de façon exponentielle lors du refroidissement ; la constante de temps est estimée par ajustement."},
 {"id": "GEO-01", "source": "Note de terrain géologie",
  "texte": "Une carte géologique dépend de l'échelle d'observation ; les sites voisins sont spatialement corrélés."},
 {"id": "GEO-02", "source": "Rapport géochimique",
  "texte": "Les teneurs en silice et en magnésium distinguent les basaltes des granites dans les échantillons analysés."},
 {"id": "STAT-01", "source": "Guide de validation",
  "texte": "La validation croisée par groupe évite qu'un même patient ou un même site apparaisse à la fois en entraînement et en test."},
 {"id": "STAT-02", "source": "Guide de validation",
  "texte": "Un intervalle de confiance par bootstrap quantifie l'incertitude d'une métrique calculée sur un petit jeu de test."},
]

import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vec = TfidfVectorizer(ngram_range=(1, 2), strip_accents='unicode', stop_words=MOTS_VIDES)
X = vec.fit_transform([d['texte'] for d in documents])

def rechercher(question, k=3):
    scores = cosine_similarity(vec.transform([question]), X).ravel()
    return [documents[i] for i in scores.argsort()[::-1][:k] if scores[i] > 0]

## 1. Le prompt RAG : contexte, tâche, contraintes, format

In [ ]:
CONSIGNES = (
    "Tu es un assistant de laboratoire. Réponds uniquement à partir des extraits fournis. "
    "Après chaque phrase, cite l'identifiant de l'extrait entre crochets, par exemple [PHY-01]. "
    "N'invente aucune valeur numérique. Si l'information manque, écris exactement : information absente du corpus."
)

def construire_prompt(question, passages):
    contexte = '\n'.join(f'[{d["id"]}] {d["texte"]}' for d in passages)
    return f'Extraits :\n{contexte}\n\nQuestion : {question}'

question = "À quelle température et sous quelle atmosphère incube-t-on les cultures, et quel risque existe-t-il ?"
passages = rechercher(question)
print(CONSIGNES, '\n'); print(construire_prompt(question, passages))

## 2. Obtenir une réponse

**Option A (facultative)** : avec une clé `ANTHROPIC_API_KEY` et le paquet `anthropic` (`pip install anthropic`), la fonction
interroge un modèle Claude. **Option B (par défaut)** : trois réponses enregistrées, dont deux fautives, servent au contrôle.

In [ ]:
import os

MODELE = 'claude-opus-5-5'   # modifiable selon votre accès et votre budget

def appeler_llm(question, passages):
    import anthropic
    client = anthropic.Anthropic()
    reponse = client.beta.messages.create(
        model=MODELE,
        max_tokens=1000,
        system=CONSIGNES,
        messages=[{'role': 'user', 'content': construire_prompt(question, passages)}],
        betas=['server-side-fallback-2026-07-01'],
        fallbacks='default',
    )
    if reponse.stop_reason == 'refusal':
        return 'information absente du corpus.'
    return ''.join(b.text for b in reponse.content if b.type == 'text')

REPONSES_SIMULEES = {
    'fidèle': "Les cultures sont incubées à 37 °C sous 5 % de CO2 [BIO-02]. "
              "Un effet de lot peut apparaître si les plaques proviennent de séries différentes [BIO-02].",
    'citation inventée': "Les cultures sont incubées à 37 °C sous 5 % de CO2 [BIO-02]. "
                         "Une validation par groupe limite les biais liés aux plaques [BIO-07].",
    'nombre inventé': "Les cultures sont incubées à 37 °C sous 5 % de CO2 pendant 48 h [BIO-02]. "
                      "Un effet de lot peut apparaître [BIO-02].",
}

if os.environ.get('ANTHROPIC_API_KEY'):
    try:
        REPONSES_SIMULEES['LLM réel'] = appeler_llm(question, passages)
    except Exception as e:
        print('Appel impossible, on garde les réponses simulées :', type(e).__name__, e)
for nom, r in REPONSES_SIMULEES.items():
    print(f'--- {nom}\n{r}\n')

## 3. Vérifier automatiquement

Trois contrôles simples, mais redoutablement efficaces :
1. chaque phrase porte **au moins une citation** ;
2. chaque citation désigne un extrait **effectivement fourni** ;
3. chaque **nombre** de la phrase figure dans un extrait cité.

Ces contrôles ne prouvent pas que la réponse est juste ; ils détectent des **défauts certains**.

In [ ]:
ids_fournis = {d['id'] for d in passages}
par_id = {d['id']: d['texte'] for d in documents}

def verifier(reponse):
    problemes = []
    phrases = [p.strip() for p in re.split(r'(?<=[.!?])\s+', reponse) if p.strip()]
    for p in phrases:
        if p.lower().startswith('information absente'):
            continue
        cites = re.findall(r'\[([A-Z]+-\d+)\]', p)
        if not cites:
            problemes.append(f'phrase sans citation : « {p} »')
            continue
        for c in cites:
            if c not in ids_fournis:
                problemes.append(f'citation {c} non fournie dans le contexte')
        texte_sources = ' '.join(par_id.get(c, '') for c in cites)
        for nombre in re.findall(r'\d+(?:[.,]\d+)?', re.sub(r'\[[^\]]*\]', '', p)):
            if nombre not in texte_sources:
                problemes.append(f'nombre {nombre} absent des sources citées')
    return problemes

for nom, r in REPONSES_SIMULEES.items():
    pb = verifier(r)
    print(f'{nom:18s} ->', 'aucun défaut détecté' if not pb else pb)

### ✏️ Exercice 1 — Un défaut que le contrôle ne voit pas

Écrivez une réponse qui **passe** les trois contrôles mais qui est pourtant **fausse** ou trompeuse. Quel contrôle supplémentaire (automatique ou humain) faudrait-il ?

In [ ]:
reponse_piege = "..."
print(verifier(reponse_piege))

<details><summary><b>Voir le corrigé</b></summary>

```python
reponse_piege = "Les cultures sont incubées à 5 °C sous 37 % de CO2 [BIO-02]."
print(verifier(reponse_piege))
```

Les deux nombres existent bien dans la source, mais ils sont **intervertis**. Il faudrait vérifier les couples (grandeur, valeur, unité), ou demander une citation littérale de la phrase source. Le contrôle humain reste indispensable.

</details>

### ✏️ Exercice 2 — Variabilité (avec clé API, facultatif)

Si vous disposez d’une clé, appelez `appeler_llm` cinq fois sur la même question et comparez les réponses : formulations, citations, défauts détectés. Sans clé, expliquez pourquoi il faut **conserver** prompts, modèle, version et date pour la traçabilité.

In [ ]:
if os.environ.get('ANTHROPIC_API_KEY'):
    for i in range(5):
        r = appeler_llm(question, passages)
        print(i, verifier(r), r[:120])
else:
    print('Pas de clé : répondre à la question de traçabilité dans une cellule de texte.')

<details><summary><b>Voir le corrigé</b></summary>

```python
if os.environ.get('ANTHROPIC_API_KEY'):
    for i in range(5):
        r = appeler_llm(question, passages)
        print(i, verifier(r), r[:120])
else:
    print('Pas de clé : répondre à la question de traçabilité dans une cellule de texte.')
```

Les sorties d’un LLM varient d’un appel et d’une version à l’autre. Sans la trace exacte (prompt, extraits, modèle, date), une analyse n’est pas reproductible.

</details>

## Interprétation scientifique (à rédiger)

1. Quels défauts la vérification automatique a-t-elle détectés ? Lesquels lui échappent ?
2. Pourquoi une citation affichée ne prouve-t-elle pas que l’affirmation est correcte ?
3. Quelles informations consigneriez-vous dans un cahier de laboratoire pour un usage de LLM ?
4. Dans votre discipline, quelles tâches confieriez-vous (ou non) à un tel assistant ?